### Ready to Go? Let's Start Your Chemical Discovery with CSU-IR for General Compound Retrieval!

💡 **Pro Tip:** Want to speed things up? You can use a powerful T4 GPU for free!

Just click on **Runtime** at the top-right of the page, select **Change runtime type**, and choose **T4 GPU** from the *Hardware accelerator* dropdown.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Final Pre-Flight Check (Crucial!)

1.  **Check Your Connection Status** at the top-right of the page.
    - Colab often connects automatically to your last-used device. If you don't see `Connecting...`, please click the **`Connect`** button manually.
2.  **Wait for the Green Checkmark** (✓) to appear next to your `RAM` and `Disk` status. This confirms you are **"Connected"**.
3.  When Colab asks for permission to access your files, go ahead and click **"Connect to Google Drive"** to authorize it.
4.  Once connected, Run Cells **"Step-by-Step"**.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### ✨ After running, get ready to see the magic unfold:

Our intelligent pipeline will automatically generate and display these exciting analytical results for you:

*   **1. Internal Stress Test**: See how the CSU-IR model performs when searching for internal samples in massive databases.
*   **2. External Benchmark**: Witness the model's true retrieval power benchmarked against the official NIST test set.
*   **3. Single raw file retrieval**: Visualize the search results of a single data.

**Note**: Due to Google Drive's memory limitations, we are only demonstrating retrieval on a smaller library in this Colab environment. For instructions on how to run a full evaluation with all libraries, please refer to the "Testing and Inference" section of the README.md in our GitHub repository.

---

### **Step 1: Setting Up Your Workspace** ⚙️

This code cell is the starting point for your discovery journey. It will automatically handle all the essential preparations, including:

1.  **Connecting to your Google Drive** to save and access data.
2.  **Cloning or updating our GitHub repository** to ensure you're using the latest code.
3.  **Installing all required Python libraries** to power the analysis.
4.  **Downloading all model weights and datasets** from Hugging Face and storing them securely in your Drive.

To execute this task, simply hover over the code cell below and click the **Run** button (▶) on the left.

In [ ]:
#@title 🧠 1: Setting Up Your Workspace
# ==============================================================================
#  CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download # Import at the top

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Use force_remount for robustness
print("✅ Google Drive mounted.")


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================
print("\n--- Defining Project Paths ---")
# Project Root
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"

# Requirements file
REQUIREMENTS_FILE = os.path.join(GDRIVE_REPO_PATH, 'requirements_colab.txt')

# --- Data and Model Destination Folders on Google Drive ---
# These are the folders where downloaded files will be saved.
DEST_LIB_GENERAL_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'processed_library', 'General')
DEST_MODEL_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'check_points')
print("✅ Paths defined.")


# ==================================================================
# --- 3. Clone or Update the 'CSU-IR' repository ---
# ==================================================================
print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your GitHub Personal Access Token in Colab Secrets (secret name: GITHUB_PAT)")

if not os.path.exists(GDRIVE_REPO_PATH):
    print(f"⏳ Cloning 'CSU-IR' to: {GDRIVE_REPO_PATH}")
    !git clone -q https://{GIT_TOKEN}@github.com/Hsqcsu/CSU-IR.git {GDRIVE_REPO_PATH}
else:
    print(f"✅ Repository already exists. Pulling latest changes...")
    !cd {GDRIVE_REPO_PATH} && git pull


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                shutil.copy(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully.")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"
PROCESSED_LIBRARY_GENERAL_TO_DOWNLOAD = {
    "Processed_Retrieval_Library/General_Retrieval/library_nist_smiles_features_fp16.gz": "library_nist_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/library_4k_smiles_features_fp16.gz": "library_4k_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/library_200k_smiles_features_fp16.gz": "library_200k_smiles_features_fp16.gz",
    #"Processed_Retrieval_Library/General_Retrieval/library_1M_smiles_features_fp16.gz": "library_1M_smiles_features_fp16.gz",
    #"Processed_Retrieval_Library/General_Retrieval/library_2M_smiles_features_fp16.gz": "library_2M_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/smiles_nist.txt": "smiles_nist.txt",
    "Processed_Retrieval_Library/General_Retrieval/smiles_4k.txt": "smiles_4k.txt",
    "Processed_Retrieval_Library/General_Retrieval/smiles_200k.txt": "smiles_200k.txt",
    #"Processed_Retrieval_Library/General_Retrieval/smiles_1M.txt": "smiles_1M.txt",
    #"Processed_Retrieval_Library/General_Retrieval/smiles_2M.txt": "smiles_2M.txt",
}
MODEL_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/CSU-IR/best_ir_model_500-4000.pth": "best_ir_model_500-4000.pth",
    "Model_weights/CSU-IR/best_smiles_model_500-4000.pth": "best_smiles_model_500-4000.pth",
}

print("✅ Download helper and file lists are ready.")

# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")
download_files_from_hf(HF_REPO_ID, PROCESSED_LIBRARY_GENERAL_TO_DOWNLOAD, DEST_LIB_GENERAL_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, MODEL_WEIGHTS_TO_DOWNLOAD, DEST_MODEL_WEIGHTS_PATH, HF_TOKEN)


# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")
print("You can now run your analysis and training notebooks.")

---
### **Step 2: Model Initialization & Data Loading** 🧠
<br>

<div style="background-color: #fffbe6; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #856404; margin-top: 0;">
        ⚠️ IMPORTANT: Please Read Before Running!
    </h4>
    <p>
        If the code cell below fails on the <strong>first run</strong> with a <code>NumPy 2.0</code> compatibility error, please do not worry. This is a known issue.
    </p>
    <p>
        <strong>Solution: Simply re-run the same cell a second time.</strong>
    </p>
    <p style="font-size: 0.9em; color: #666;">
        <em><strong>Technical Reason:</strong> You might see an error like <code>A module that was compiled using NumPy 1.x cannot be run in NumPy 2.0.x</code>. This happens because some pre-compiled packages (like rdkit) need the environment to adjust to the new NumPy version. The second run typically resolves this dependency conflict.</em>
    </p>
</div>


In [ ]:
#@title 🧠 2. Global Initialization (Load All Models & Data)
import sys
import os
import gzip
import torch
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from rdkit.Chem import Draw, MolFromSmiles
from rdkit import Chem
from google.colab import files
import pandas as pd
import jcamp
import numpy as np
from PIL import Image
import io

# --- 1. Set up project paths ---
print("--- Setting up project paths for module imports ---")
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"
CSU_IR_MODULE_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR')
if CSU_IR_MODULE_PATH not in sys.path: sys.path.insert(0, CSU_IR_MODULE_PATH)
print("✅ Project paths set.")

# --- 2. Import custom modules ---
print("\n--- Importing custom modules ---")
from model.IR_encoder import IRModel
from model.SMILES_encoder import SmilesModel
from data_process.ir_process import preprocess_spectra_higer_500, preprocess_spectra_lower_500
from test_and_infer.infer import get_topK_result
from test_and_infer.infer import normalize_smiles
print("✅ Custom modules imported.")

# --- 3. Define all file paths ---
print("\n--- Defining all file paths ---")
TOKENIZER_PATH = os.path.join(CSU_IR_MODULE_PATH, "model", "tokenizer-smiles-roberta-1e_new")
PRETRAIN_SMILES_MODEL_PATH = os.path.join(DEST_MODEL_WEIGHTS_PATH, "best_smiles_model_500-4000.pth")
PRETRAIN_IR_MODEL_PATH = os.path.join(DEST_MODEL_WEIGHTS_PATH, "best_ir_model_500-4000.pth")
TEST_DATA_PATH = os.path.join(CSU_IR_MODULE_PATH, "data", "test_data")
SINGLE_TEST_DATA_PATH = os.path.join(CSU_IR_MODULE_PATH, "data", "example_library_and_ir_for_user_dinfined","Ethanol.jdx")
print("✅ File paths defined.")

# --- 4. Initialize all models and load weights ---
print("\n--- Initializing and loading all models ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Retrieval Models
ir_retrieval_model = IRModel()
smiles_retrieval_model = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)
ir_retrieval_model.load_state_dict(torch.load(PRETRAIN_IR_MODEL_PATH, map_location=device))
smiles_retrieval_model.load_state_dict(torch.load(PRETRAIN_SMILES_MODEL_PATH, map_location=device))
ir_retrieval_model.to(device).eval()
smiles_retrieval_model.to(device).eval()
print("✅ Retrieval models loaded.")

# Classifier Models
# Note: The encoders for classifiers are separate instances
ir_classifier_encoder = IRModel()
smiles_classifier_encoder = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)


ir_classifier_encoder.to(device).eval()
smiles_classifier_encoder.to(device).eval()
print("✅ Classifier models loaded.")

# --- 5. Define helper functions ---
def load_gz_features(path):
    with gzip.open(path, 'rb') as f: return torch.load(f).to(torch.float32)
def load_smiles_list(path):
    with open(path, 'r', encoding='utf-8') as f: return [line.strip() for line in f if line.strip()]
print("✅ Helper functions defined.")

# --- 6. Load all feature libraries and SMILES lists into memory ---
print("\n--- Loading all feature libraries (this may take a moment) ---")
# General Libraries
library_nist_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_nist_smiles_features_fp16.gz'))
library_4k_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_4k_smiles_features_fp16.gz'))
library_200k_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_200k_smiles_features_fp16.gz'))
#library_1M_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_1M_smiles_features_fp16.gz'))
#library_2M_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_2M_smiles_features_fp16.gz'))

general_smiles_nist = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_nist.txt'))
general_smiles_4k = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_4k.txt'))
general_smiles_200k = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_200k.txt'))
#general_smiles_1M = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_1M.txt'))
#general_smiles_2M = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_2M.txt'))
print("✅ All libraries loaded.")

print("\n\n🎉🎉🎉 Global initialization complete! System is ready! 🎉🎉🎉")

In [ ]:
#@title 🔬 Task 1: Internal Test Set Bacth Retrieval
#@markdown This test evaluates the model's performance by searching for samples from our internal test set against various general-purpose libraries.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search = "General 4k Library" #@param ["General 4k Library", "General 200k Library"]

# --- Map user selection to actual data ---
library_map = {
    "General 4k Library": (library_4k_features, general_smiles_4k),
    "General 200k Library": (library_200k_features, general_smiles_200k),
    #"General 1M Library": (library_1M_features, general_smiles_1M),
    #"General 2M Library": (library_2M_features, general_smiles_2M)
}
selected_features, selected_smiles = library_map[library_to_search]

class IRSmilesDataset(Dataset):
    def __init__(self, ir_spectra, smiles):
        self.ir_spectra = ir_spectra
        self.smiles = smiles

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        return self.ir_spectra[idx], self.smiles[idx]

# --- Define evaluation function ---
def batch_evaluate_retrieval(loader, features_db, smiles_db, name):
    top_1, top_5, top_10, total = 0, 0, 0, 0
    with torch.no_grad():
        for ir_batch, smiles_batch in tqdm(loader, desc=f"Evaluating {name}"):
            ir_features = ir_retrieval_model(ir_batch.to(device))
            for i, ir_feature in enumerate(ir_features):
                original_smiles = normalize_smiles(smiles_batch[i])
                indices, _ = get_topK_result(ir_feature.unsqueeze(0), features_db, 10)
                for rank, lib_idx in enumerate(indices[0]):
                    if lib_idx < len(smiles_db) and original_smiles == normalize_smiles(smiles_db[lib_idx]):
                        if rank == 0: top_1 += 1
                        if rank < 5: top_5 += 1
                        if rank < 10: top_10 += 1
                        break
                total += 1
    print(f"\n--- Results for {name} ---\nTop-1: {top_1/total:.4f} ({top_1}/{total})| Top-5: {top_5/total:.4f} | Top-10: {top_10/total:.4f}")


# --- Load internal test data and run ---
Internal_test_smiles, Internal_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "Internal_test", "Internal_test_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "Internal_test", "Internal_test_ir.pt"))
Internal_test_loader = DataLoader(IRSmilesDataset(Internal_test_ir, Internal_test_smiles), batch_size=208, shuffle=False)
batch_evaluate_retrieval(Internal_test_loader, selected_features, selected_smiles, f"Internal Test in {library_to_search}")

In [ ]:
#@title 🌍 Task 2: External NIST Test Set Bacth Retrieval
#@markdown This test benchmarks the model against the official NIST test set, searching in various general-purpose libraries.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search_nist = "General NIST Library" #@param ["General NIST Library", "General 4k Library", "General 200k Library"]

# --- Map user selection to actual data ---
library_map_nist = {
    "General NIST Library": (library_nist_features, general_smiles_nist),
    "General 4k Library": (library_4k_features, general_smiles_4k),
    "General 200k Library": (library_200k_features, general_smiles_200k),
    #"General 1M Library": (library_1M_features, general_smiles_1M),
    #"General 2M Library": (library_2M_features, general_smiles_2M)
}
selected_features_nist, selected_smiles_nist = library_map_nist[library_to_search_nist]

# --- Load NIST test data and run ---
External_nist_test_smiles, External_nist_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "External_nist_test", "nist_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "External_nist_test", "nist_ir.pt"))
External_nist_test_loader = DataLoader(IRSmilesDataset(External_nist_test_ir, External_nist_test_smiles), batch_size=208, shuffle=False)
batch_evaluate_retrieval(External_nist_test_loader, selected_features_nist, selected_smiles_nist, f"External NIST Test in {library_to_search_nist}")

In [ ]:
#@title 🌍 Task 3: Single IR Retrieval. This test Ethanol retrieval in various general-purpose libraries as an example, the IR spectrum is taken from Nist.
#@markdown This test Ethanol retrieval in various general-purpose libraries as an example, the IR spectrum is taken from Nist.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search_nist = "General NIST Library" #@param ["General NIST Library", "General 4k Library", "General 200k Library"]

# --- Map user selection to actual data ---
library_map_nist = {
    "General NIST Library": (library_nist_features, general_smiles_nist),
    "General 4k Library": (library_4k_features, general_smiles_4k),
    "General 200k Library": (library_200k_features, general_smiles_200k),
    #"General 1M Library": (library_1M_features, general_smiles_1M),
    #"General 2M Library": (library_2M_features, general_smiles_2M)
}
selected_features_nist, selected_smiles_nist = library_map_nist[library_to_search_nist]

def draw_molecules(smiles_list, scores_list):
    if len(smiles_list) != len(scores_list):
        raise ValueError("smiles_list and scores_list must be the same length!")
    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    legends = [f"Top {i + 1}: {round(score, 4)}" for i, score in enumerate(scores_list)]
    valid_mols_legends = [(mol, legend) for mol, legend in zip(mols, legends) if mol is not None]
    valid_mols, valid_legends = zip(*valid_mols_legends)
    img = Draw.MolsToGridImage(
        valid_mols,
        molsPerRow=5,
        subImgSize=(300, 300),
        legends=list(valid_legends)
    )
    return img

def single_retrieval_library(ir_feature, library_features, library_smiles):
    indices, scores = get_topK_result(ir_feature, library_features, 10)

    top_smiles = []
    top_scores = []

    for sco, idx in zip(scores[0], indices[0]):
        retrieved_smiles = library_smiles[idx.item()]
        top_smiles.append(retrieved_smiles)
        top_scores.append(sco.item())

    img = draw_molecules(top_smiles,top_scores)
    return img, top_scores, top_smiles

def single_retrieval(ir_spectra_file, spectrum_type, library_features, library_smiles):
    if hasattr(ir_spectra_file, 'name'):
        file_path = ir_spectra_file.name
    else:
        file_path = ir_spectra_file
    print(f"Processing file: {file_path}")
    if file_path.lower().endswith('.csv'):
        df = pd.read_csv(file_path)
        wavenumbers = df.iloc[:, 0].values
        transmittances = df.iloc[:, 1].values
    elif file_path.lower().endswith('.jdx'):
        data = jcamp.jcamp_readfile(file_path)
        wavenumbers = np.array(data['x'], dtype=float)
        transmittances = np.array(data['y'], dtype=float)
    else:
        raise ValueError("Unsupported file format. Please upload a CSV or JDX file.")

    if spectrum_type == "absorbance spectrum":
        if wavenumbers[0] > 500:
            ir_data = preprocess_spectra_higer_500(wavenumbers, transmittances)
        else:
            ir_data = preprocess_spectra_lower_500(file_path, wavenumbers, transmittances)
    else:
        transmittances = transmittances / 100.0
        if wavenumbers[0] > 500:
            ir_data = preprocess_spectra_higer_500(wavenumbers, transmittances)
        else:
            ir_data = preprocess_spectra_lower_500(file_path, wavenumbers, transmittances)

    ir_spectra_tensor = torch.tensor(ir_data, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        ir_feature = ir_retrieval_model(ir_spectra_tensor)

    img, scores,top_smiles = single_retrieval_library(ir_feature, library_features, library_smiles)
    return img, scores,top_smiles

ir_spectra_file = SINGLE_TEST_DATA_PATH
img, scores, top_smiles = single_retrieval(ir_spectra_file , spectrum_type="absorbance spectrum", library_features = selected_features_nist, library_smiles = selected_smiles_nist)

# --- Display the results ---
if img:
    print("\n--- Retrieval Results ---")
    for i, (smile, score) in enumerate(zip(top_smiles, scores)):
        print(f"Top {i+1}: Score={score:.4f}, SMILES={smile}")

    print("\n--- Top 10 Candidate Molecules ---")
    # By placing the image object as the last line of the cell, Colab will automatically display it.
    display(img)
else:
    print("\nNo image to display. Please run the cell again and provide a file.")